<a href="https://colab.research.google.com/github/rakshitarajan/CreditCardFraudDetection/blob/main/CreditCardFraudDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Fraud Detection using Random Forest - PaySim Dataset


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score
)

# 1. Load Dataset


df = pd.read_csv("PS_20174392719_1491204439457_log.csv")

print("Dataset shape:", df.shape)
print(df.head())


# 2. Basic Data Analysis

print("\nMissing values:")
print(df.isnull().sum())

print("\nClass distribution:")
print(df["isFraud"].value_counts())


# 3. Remove Unnecessary Columns

# nameOrig and nameDest are transaction IDs.
# They generally don't provide useful information for the model.

df = df.drop(["nameOrig", "nameDest"], axis=1)


# 4. Encode Transaction Type

encoder = LabelEncoder()

df["type"] = encoder.fit_transform(df["type"])


# 5. Feature Engineering

# Difference between sender's balance before and after transaction
df["balance_change_orig"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

# Difference between receiver's balance before and after transaction
df["balance_change_dest"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

# Check whether the transaction amount is unusually large
df["amount_to_balance_ratio"] = (
    df["amount"] / (df["oldbalanceOrg"] + 1)
)


# 6. Separate Features and Target

X = df.drop("isFraud", axis=1)
y = df["isFraud"]


# 7. Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape)
print("Testing samples:", X_test.shape)


# 8. Random Forest Model


model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


# 9. Train Model


print("\nTraining Random Forest...")

model.fit(X_train, y_train)

print("Training completed.")


# 10. Predictions

y_pred = model.predict(X_test)

# Probability of transaction being fraudulent
y_prob = model.predict_proba(X_test)[:, 1]


# 11. Evaluation

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob))


# 12. Feature Importance

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance:")
print(feature_importance)


# 13. Predict a New Transaction


new_transaction = pd.DataFrame({
    "step": [1],
    "type": [encoder.transform(["TRANSFER"])[0]],
    "amount": [100000],
    "oldbalanceOrg": [100000],
    "newbalanceOrig": [0],
    "oldbalanceDest": [50000],
    "newbalanceDest": [150000],
    "balance_change_orig": [100000],
    "balance_change_dest": [100000],
    "amount_to_balance_ratio": [0.99999]
})

prediction = model.predict(new_transaction)
probability = model.predict_proba(new_transaction)[0][1]

if prediction[0] == 1:
    print("\n🚨 FRAUDULENT TRANSACTION")
else:
    print("\n✅ LEGITIMATE TRANSACTION")

print(f"Fraud probability: {probability:.2%}")